# deploy

> Serve an export behind an HTTP endpoint, in a container on this machine or as a SageMaker endpoint, with one call; and write the infrastructure for it as a CloudFormation template or a CDK app.

In [ ]:
#| default_exp deploy

In [ ]:
#| export
import io, json, os, re, shutil, subprocess, sys, tarfile, tempfile, time, urllib.request
from pathlib import Path

import sysone
from sysone.core import dispatch
from sysone.cloud import Backend, Local, SageMaker, CPU, GPU, Neuron, Hardware, machine, hardware, backend
from sysone.aws import aws_client, bucket, role_arn, image_for, image_tag, emulated

In [ ]:
#| hide
from fastcore.test import test_eq, test_fail, test_ne
import logging
logging.getLogger("transformers").setLevel(logging.ERROR)
from sysone.aws import floci_up, build_image
from sysone.data import TypedDecisions
from sysone.learner import decision_learner
from sysone.inference import Decider

An export is served by sysone's server (`sysone serve`, see [inference](06_inference.ipynb#serving)), which [aws](48_aws.ipynb) packs in a container image with SageMaker's routes. `deploy` runs that image:

- `deploy(export, on="local")` runs it in Docker on this machine, with the export mounted as the model: the same container a SageMaker endpoint runs, to try before deploying;
- `deploy(export, on=SageMaker(...), hw="cpu")` uploads the export as `model.tar.gz` and creates the model, the endpoint configuration and the endpoint (with `serverless=` for a serverless endpoint), then waits for it to be in service.

Both return an `Endpoint`: `predict(state, questions)` answers as `Decider` does, and `delete()` removes what `deploy` made. As with jobs, `deploy` is a generic function on the backend, so a backend's module adds its own method.

In [ ]:
#| export
class Endpoint:
    "A served model: `predict` answers as `Decider` does, `delete` removes what was deployed"
    name:str
    def invoke(self, body): raise NotImplementedError
    def predict(self, state, questions:dict) -> dict:
        "The answers to `questions` about `state`, in the Jev schema"
        return self.invoke({"state": state, "questions": questions})["answers"]
    def predict_many(self, requests:list) -> list:
        "Answers for several `{state, questions}` requests in one call"
        return [r["answers"] for r in self.invoke(list(requests))]
    def __repr__(self): return f"{type(self).__name__}({self.name})"

def model_tarball(export) -> bytes:
    "An export folder as SageMaker's model.tar.gz: its files at the archive's root"
    export, buf = Path(export), io.BytesIO()
    if not (export / "sysone.json").exists(): raise FileNotFoundError(f"{export} is not a sysone export: it has no sysone.json")
    with tarfile.open(fileobj=buf, mode="w:gz") as t:
        for p in sorted(export.rglob("*")):
            if p.is_file() and p.name != "sysone_outputs.tar.gz": t.add(p, arcname=str(p.relative_to(export)))
    return buf.getvalue()

def endpoint_name(export, name:str|None=None) -> str:
    "A valid endpoint name (letters, digits, hyphens, at most 63): `name`, or the export's folder name and a time"
    s = name or f"{Path(export).name}-{time.strftime('%Y%m%d-%H%M%S')}"
    s = re.sub(r"[^A-Za-z0-9-]+", "-", s).strip("-")[:63]
    if not s: raise ValueError("an endpoint needs a name of letters, digits and hyphens")
    return s

@dispatch
def deploy_to(b:Backend, export:Path, hw:Hardware, name:str, image:str|None, **kw) -> Endpoint:
    "Serve `export` on backend `b` (a backend's module adds its method)"
    raise NotImplementedError(f"{b.name} cannot serve models")

def deploy(export, on="local", hw=None, name:str|None=None, image:str|None=None, **kw) -> Endpoint:
    "Serve an export: in Docker here (`on='local'`) or as a SageMaker endpoint (`on=SageMaker(...)`); returns its `Endpoint`"
    b = backend(on)
    hw = hardware(hw) if hw is not None else CPU()
    return deploy_to(b, Path(export), hw, endpoint_name(export, name), image, **kw)

In [ ]:
test_eq(endpoint_name("/tmp/my_jev v2", "my_jev v2"), "my-jev-v2")
test_fail(lambda: model_tarball(tempfile.mkdtemp()), contains="no sysone.json")

## In Docker, here

The container runs `serve` with the export mounted read-only at `/opt/ml/model`, on a port of `127.0.0.1` only. `LocalEndpoint.delete()` removes the container.

In [ ]:
#| export
def _free_port() -> int:
    import socket
    with socket.socket() as s: s.bind(("127.0.0.1", 0)); return s.getsockname()[1]

def _post(url:str, body, timeout:float=120):
    req = urllib.request.Request(url, data=json.dumps(body).encode(), headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as r: return json.loads(r.read())

class LocalEndpoint(Endpoint):
    "sysone's image serving an export in a Docker container of this machine"
    def __init__(self, name:str, container:str, url:str): self.name, self.container, self.url = name, container, url
    def invoke(self, body): return _post(self.url + "/invocations", body)
    def delete(self): subprocess.run(["docker", "rm", "-f", self.container], capture_output=True)

@deploy_to.register
def deploy_local(b:Local, export:Path, hw:Hardware, name:str, image:str|None, port:int|None=None, timeout:float=180.0) -> LocalEndpoint:
    image, port = image or image_tag(isinstance(hw, GPU)), port or _free_port()
    container = f"sysone-endpoint-{name}"[:63]
    gpu = ["--gpus", "all"] if isinstance(hw, GPU) else []
    subprocess.run(["docker", "run", "-d", "--name", container, "-p", f"127.0.0.1:{port}:8080", "-v", f"{Path(export).resolve()}:/opt/ml/model:ro"]
                   + gpu + [image, "serve"], check=True, capture_output=True)
    url, t0 = f"http://127.0.0.1:{port}", time.time()
    while True:
        try:
            if urllib.request.urlopen(url + "/ping", timeout=2).status == 200: return LocalEndpoint(name, container, url)
        except Exception: pass
        if time.time() - t0 > timeout:
            logs = subprocess.run(["docker", "logs", "--tail", "30", container], capture_output=True, text=True)
            subprocess.run(["docker", "rm", "-f", container], capture_output=True)
            raise RuntimeError(f"the endpoint did not answer /ping within {timeout:.0f} s:\n{logs.stdout + logs.stderr}")
        time.sleep(1)

## On SageMaker

`deploy_sagemaker` uploads `model.tar.gz` under `sysone/models/<name>/`, then creates the three resources with the same name. On AWS the image must be in ECR (build it with `build_image(platform="linux/amd64")` and push it); on an emulator the local tag runs as it is. `serverless={"MemorySizeInMB": 4096, "MaxConcurrency": 5}` makes a serverless endpoint, which costs nothing between requests but starts cold. `SageMakerEndpoint.delete()` removes the endpoint, its configuration and the model.

In [ ]:
#| export
class SageMakerEndpoint(Endpoint):
    "A SageMaker endpoint serving an export"
    def __init__(self, name:str, b:SageMaker): self.name, self.b = name, b
    def invoke(self, body):
        r = aws_client(self.b, "sagemaker-runtime").invoke_endpoint(EndpointName=self.name, ContentType="application/json",
                                                                      Accept="application/json", Body=json.dumps(body))
        return json.loads(r["Body"].read())
    def status(self) -> str: return aws_client(self.b, "sagemaker").describe_endpoint(EndpointName=self.name)["EndpointStatus"]
    def delete(self):
        sm = aws_client(self.b, "sagemaker")
        for call, kw in ((sm.delete_endpoint, {"EndpointName": self.name}), (sm.delete_endpoint_config, {"EndpointConfigName": self.name}),
                         (sm.delete_model, {"ModelName": self.name})):
            try: call(**kw)
            except Exception: pass

def endpoint_requests(name:str, image:str, model_data:str, role:str, instance:str|None, serverless:dict|None=None,
                      env:dict|None=None) -> dict:
    "The CreateModel, CreateEndpointConfig and CreateEndpoint requests for an endpoint"
    variant = {"VariantName": "main", "ModelName": name}
    variant |= {"ServerlessConfig": serverless} if serverless else {"InstanceType": instance, "InitialInstanceCount": 1}
    return {"model": {"ModelName": name, "ExecutionRoleArn": role,
                      "PrimaryContainer": {"Image": image, "ModelDataUrl": model_data, "Environment": env or {}}},
            "config": {"EndpointConfigName": name, "ProductionVariants": [variant]},
            "endpoint": {"EndpointName": name, "EndpointConfigName": name}}

@deploy_to.register
def deploy_sagemaker(b:SageMaker, export:Path, hw:Hardware, name:str, image:str|None, serverless:dict|None=None,
                     wait:bool=True, timeout:float=1800.0) -> SageMakerEndpoint:
    inst = None if serverless else machine(b, hw)
    bk, key = bucket(b), f"sysone/models/{name}/model.tar.gz"
    aws_client(b, "s3").put_object(Bucket=bk, Key=key, Body=model_tarball(export))
    req = endpoint_requests(name, image or image_for(b, inst or "ml.m5.xlarge"), f"s3://{bk}/{key}", role_arn(b), inst, serverless,
                            env={"SYSONE_DEVICE": "cuda" if isinstance(hw, GPU) else "cpu"})
    sm = aws_client(b, "sagemaker")
    sm.create_model(**req["model"]); sm.create_endpoint_config(**req["config"]); sm.create_endpoint(**req["endpoint"])
    ep, t0 = SageMakerEndpoint(name, b), time.time()
    while wait:
        st = ep.status()
        if st == "InService": break
        if st == "Failed" or time.time() - t0 > timeout:
            reason = sm.describe_endpoint(EndpointName=name).get("FailureReason")
            raise RuntimeError(f"endpoint {name} is {st}: {reason}")
        time.sleep(3)
    return ep

In [ ]:
rq = endpoint_requests("my-jev", "sysone:0.3.0-cpu", "s3://b/m.tar.gz", "arn:aws:iam::1:role/r", "ml.m5.xlarge")
test_eq(rq["config"]["ProductionVariants"][0]["InstanceType"], "ml.m5.xlarge")
rs = endpoint_requests("my-jev", "img", "s3://b/m.tar.gz", "arn:r", None, serverless={"MemorySizeInMB": 4096, "MaxConcurrency": 5})
test_eq(("InstanceType" in rs["config"]["ProductionVariants"][0], rs["config"]["ProductionVariants"][0]["ServerlessConfig"]["MemorySizeInMB"]), (False, 4096))

## Infrastructure as code

An endpoint made by hand is hard to reproduce, review or delete cleanly; a template is a file in the repository. `cloudformation` writes one: an execution role allowed to read the model and pull the image (and nothing else), the model, the endpoint configuration and the endpoint, and on request autoscaling on invocations per instance and an MLflow tracking server with its own role and bucket. The image, the model's S3 URI, the instance type and count are parameters, so the same template serves every model.

In [ ]:
#| export
def cloudformation(name:str, instance:str="ml.m5.xlarge", count:int=1, serverless:dict|None=None, autoscale:tuple|None=None,
                   mlflow:bool=False, description:str|None=None) -> dict:
    "A CloudFormation template (as a dict) for a sysone endpoint: role, model, endpoint config, endpoint, optional autoscaling and MLflow"
    sub = lambda s: {"Fn::Sub": s}
    t = {"AWSTemplateFormatVersion": "2010-09-09", "Description": description or f"sysone endpoint {name}",
         "Parameters": {"ImageUri": {"Type": "String", "Description": "sysone's image in ECR"},
                        "ModelDataUrl": {"Type": "String", "Description": "s3://.../model.tar.gz of the export"},
                        "InstanceType": {"Type": "String", "Default": instance}, "InstanceCount": {"Type": "Number", "Default": count}},
         "Resources": {}, "Outputs": {}}
    R = t["Resources"]
    R["ExecutionRole"] = {"Type": "AWS::IAM::Role", "Properties": {
        "AssumeRolePolicyDocument": {"Version": "2012-10-17", "Statement": [{"Effect": "Allow", "Principal": {"Service": "sagemaker.amazonaws.com"}, "Action": "sts:AssumeRole"}]},
        "Policies": [{"PolicyName": "sysone-endpoint", "PolicyDocument": {"Version": "2012-10-17", "Statement": [
            {"Effect": "Allow", "Action": ["s3:GetObject"], "Resource": sub("arn:${AWS::Partition}:s3:::*/sysone/models/*")},
            {"Effect": "Allow", "Action": ["ecr:GetAuthorizationToken", "ecr:BatchGetImage", "ecr:GetDownloadUrlForLayer"], "Resource": "*"},
            {"Effect": "Allow", "Action": ["logs:CreateLogGroup", "logs:CreateLogStream", "logs:PutLogEvents", "cloudwatch:PutMetricData"], "Resource": "*"}]}}]}}
    R["Model"] = {"Type": "AWS::SageMaker::Model", "Properties": {"ModelName": name, "ExecutionRoleArn": {"Fn::GetAtt": ["ExecutionRole", "Arn"]},
        "PrimaryContainer": {"Image": {"Ref": "ImageUri"}, "ModelDataUrl": {"Ref": "ModelDataUrl"}, "Environment": {"SYSONE_DEVICE": "cpu"}}}}
    variant = {"VariantName": "main", "ModelName": {"Fn::GetAtt": ["Model", "ModelName"]}}
    variant |= {"ServerlessConfig": serverless} if serverless else {"InstanceType": {"Ref": "InstanceType"}, "InitialInstanceCount": {"Ref": "InstanceCount"}}
    R["EndpointConfig"] = {"Type": "AWS::SageMaker::EndpointConfig", "Properties": {"ProductionVariants": [variant]}}
    R["Endpoint"] = {"Type": "AWS::SageMaker::Endpoint", "Properties": {"EndpointName": name,
        "EndpointConfigName": {"Fn::GetAtt": ["EndpointConfig", "EndpointConfigName"]}}}
    t["Outputs"]["EndpointName"] = {"Value": {"Fn::GetAtt": ["Endpoint", "EndpointName"]}}
    if autoscale and not serverless:
        lo, hi, per_instance = autoscale
        rid = f"endpoint/{name}/variant/main"
        R["ScalableTarget"] = {"Type": "AWS::ApplicationAutoScaling::ScalableTarget", "DependsOn": "Endpoint", "Properties": {
            "MinCapacity": lo, "MaxCapacity": hi, "ResourceId": rid, "ScalableDimension": "sagemaker:variant:DesiredInstanceCount",
            "ServiceNamespace": "sagemaker"}}
        R["ScalingPolicy"] = {"Type": "AWS::ApplicationAutoScaling::ScalingPolicy", "Properties": {
            "PolicyName": f"{name}-invocations", "PolicyType": "TargetTrackingScaling", "ScalingTargetId": {"Ref": "ScalableTarget"},
            "TargetTrackingScalingPolicyConfiguration": {"TargetValue": per_instance,
                "PredefinedMetricSpecification": {"PredefinedMetricType": "SageMakerVariantInvocationsPerInstance"}}}}
    if mlflow:
        R["MlflowBucket"] = {"Type": "AWS::S3::Bucket", "Properties": {"BucketEncryption": {"ServerSideEncryptionConfiguration": [
            {"ServerSideEncryptionByDefault": {"SSEAlgorithm": "AES256"}}]}}}
        R["MlflowRole"] = {"Type": "AWS::IAM::Role", "Properties": {
            "AssumeRolePolicyDocument": {"Version": "2012-10-17", "Statement": [{"Effect": "Allow", "Principal": {"Service": "sagemaker.amazonaws.com"}, "Action": "sts:AssumeRole"}]},
            "Policies": [{"PolicyName": "sysone-mlflow", "PolicyDocument": {"Version": "2012-10-17", "Statement": [
                {"Effect": "Allow", "Action": ["s3:GetObject", "s3:PutObject", "s3:ListBucket", "s3:DeleteObject"],
                 "Resource": [{"Fn::GetAtt": ["MlflowBucket", "Arn"]}, sub("${MlflowBucket.Arn}/*")]}]}}]}}
        R["Mlflow"] = {"Type": "AWS::SageMaker::MlflowTrackingServer", "Properties": {"TrackingServerName": f"{name}-mlflow"[:63],
            "ArtifactStoreUri": sub("s3://${MlflowBucket}/mlflow"), "RoleArn": {"Fn::GetAtt": ["MlflowRole", "Arn"]}}}
        t["Outputs"]["TrackingServerArn"] = {"Value": {"Fn::GetAtt": ["Mlflow", "TrackingServerArn"]}}
    return t

def write_template(t:dict, path) -> Path:
    "Write a template as JSON (`.json`) or YAML (anything else)"
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    if path.suffix == ".json": path.write_text(json.dumps(t, indent=2))
    else:
        import yaml
        path.write_text(yaml.safe_dump(t, sort_keys=False))
    return path

In [ ]:
t = cloudformation("my-jev", autoscale=(1, 4, 100.0), mlflow=True)
test_eq(sorted(r["Type"] for r in t["Resources"].values()),
        sorted(["AWS::IAM::Role", "AWS::SageMaker::Model", "AWS::SageMaker::EndpointConfig", "AWS::SageMaker::Endpoint",
                "AWS::ApplicationAutoScaling::ScalableTarget", "AWS::ApplicationAutoScaling::ScalingPolicy", "AWS::S3::Bucket",
                "AWS::IAM::Role", "AWS::SageMaker::MlflowTrackingServer"]))
ts = cloudformation("my-jev", serverless={"MemorySizeInMB": 4096, "MaxConcurrency": 5})
test_eq(list(ts["Resources"]["EndpointConfig"]["Properties"]["ProductionVariants"][0]), ["VariantName", "ModelName", "ServerlessConfig"])
print(write_template(t, Path(tempfile.mkdtemp()) / "endpoint.yaml").read_text()[:900])

The same infrastructure as a CDK app: `cdk_app` writes a Python CDK project (`app.py`, `cdk.json`, `requirements.txt`) whose stack holds the template's resources as CDK's L1 constructs, for teams that keep infrastructure in CDK; `cdk_synth` turns it into CloudFormation without an AWS account (it needs Node.js and the `cdk` CLI, and fetches `aws-cdk-lib` into a throwaway environment with `uv`).

In [ ]:
#| export
CDK_APP = '''#!/usr/bin/env python3
# A sysone endpoint as a CDK app, written by sysone.deploy.cdk_app: `cdk synth`, then `cdk deploy`
import json
from pathlib import Path
import aws_cdk as cdk

class SysoneEndpoint(cdk.Stack):
    "The resources of sysone's CloudFormation template, as CDK constructs (cdk.CfnInclude keeps them one-to-one)"
    def __init__(self, scope, id, **kw):
        super().__init__(scope, id, **kw)
        from aws_cdk import cloudformation_include as cfn_inc
        cfn_inc.CfnInclude(self, "Template", template_file=str(Path(__file__).parent / "template.json"),
                           parameters={{"ImageUri": "{image}", "ModelDataUrl": "{model_data}"}})

app = cdk.App()
SysoneEndpoint(app, "{stack}")
app.synth()
'''

def cdk_app(dest, name:str, image:str, model_data:str, **kw) -> Path:
    "Write a Python CDK app for a sysone endpoint (the `cloudformation` template, included as CDK constructs)"
    dest = Path(dest); dest.mkdir(parents=True, exist_ok=True)
    write_template(cloudformation(name, **kw), dest / "template.json")
    stack = re.sub(r"[^A-Za-z0-9-]", "-", f"sysone-{name}")
    (dest / "app.py").write_text(CDK_APP.format(image=image, model_data=model_data, stack=stack))
    (dest / "cdk.json").write_text(json.dumps({"app": "python3 app.py"}, indent=2))
    (dest / "requirements.txt").write_text("aws-cdk-lib>=2.272\nconstructs>=10\n")
    return dest

def cdk_synth(app_dir, timeout:float=600) -> dict:
    "Synthesize a CDK app to CloudFormation, offline (Node.js and the cdk CLI; aws-cdk-lib in a throwaway uv environment); returns the template"
    app_dir = Path(app_dir)
    if shutil.which("cdk") is None: raise RuntimeError("the CDK CLI is not installed: npm install -g aws-cdk")
    py = "uv run --no-project --with aws-cdk-lib --with constructs python3 app.py" if shutil.which("uv") else "python3 app.py"
    r = subprocess.run(["cdk", "synth", "--app", py, "--output", "cdk.out", "--quiet"], cwd=app_dir, capture_output=True, text=True, timeout=timeout,
                       env=os.environ | {"CDK_DISABLE_VERSION_CHECK": "1", "JSII_SILENCE_WARNING_UNTESTED_NODE_VERSION": "1"})
    if r.returncode != 0: raise RuntimeError(f"cdk synth failed: {(r.stdout + r.stderr)[-2000:]}")
    out = sorted((app_dir / "cdk.out").glob("*.template.json"))
    return json.loads(out[0].read_text())

In [ ]:
app_dir = cdk_app(Path(tempfile.mkdtemp()) / "endpoint-cdk", "my-jev", "123456789012.dkr.ecr.us-east-1.amazonaws.com/sysone:0.3.0-cpu", "s3://b/sysone/models/my-jev/model.tar.gz")
compile((app_dir / "app.py").read_text(), "app.py", "exec")
test_eq(sorted(p.name for p in app_dir.iterdir()), ["app.py", "cdk.json", "requirements.txt", "template.json"])

## On floci and Docker

These cells need Docker and the image (`sysone.aws.build_image()`); they are flagged `floci`. They deploy the tiny export in a container here and as a SageMaker endpoint on floci, ask both the same question, deploy the CloudFormation template to floci (which makes the role and accepts the SageMaker resources as stand-ins), and synthesize the CDK app (which fetches `aws-cdk-lib`, about 60 MB, once).

In [ ]:
#| floci
url = floci_up(); build_image()
data = TypedDecisions.from_json("fixtures/typed_decisions_tiny.jsonl", valid=0.25, calib=0.2)
learn = decision_learner(data, "fixtures/tiny-modernbert", train="head", preset="cpu", logging_steps=1, disable_tqdm=True)
learn.fit(1, lr=1e-3)
exp = learn.export(Path(tempfile.mkdtemp()) / "tiny-jev")
rec = json.loads(Path("fixtures/typed_decisions_tiny.jsonl").read_text().splitlines()[0])
want = json.loads(json.dumps(Decider.load(exp, device="cpu").predict(rec["state"], rec["questions"])))
local = deploy(exp, on="local")
test_eq(local.predict(rec["state"], rec["questions"]), want)
test_eq(local.predict_many([{"state": rec["state"], "questions": rec["questions"]}] * 2), [want] * 2)
local.delete()

In [ ]:
#| floci
sm = deploy(exp, on=SageMaker(endpoint_url=url), name="tiny-jev")
test_eq(sm.status(), "InService")
test_eq(sm.predict(rec["state"], rec["questions"]), want)
sm.delete()

In [ ]:
#| floci
cfn = aws_client(SageMaker(endpoint_url=url), "cloudformation")
cfn.create_stack(StackName="sysone-tiny-jev", TemplateBody=json.dumps(cloudformation("tiny-jev-cfn")), Capabilities=["CAPABILITY_IAM"],
                 Parameters=[{"ParameterKey": "ImageUri", "ParameterValue": image_tag()}, {"ParameterKey": "ModelDataUrl", "ParameterValue": "s3://sysone/x/model.tar.gz"}])
for _ in range(60):
    st = cfn.describe_stacks(StackName="sysone-tiny-jev")["Stacks"][0]["StackStatus"]
    if not st.endswith("IN_PROGRESS"): break
    time.sleep(1)
test_eq(st, "CREATE_COMPLETE")
cfn.delete_stack(StackName="sysone-tiny-jev")

In [ ]:
#| floci
synth = cdk_synth(app_dir)
assert {"AWS::SageMaker::Model", "AWS::SageMaker::EndpointConfig", "AWS::SageMaker::Endpoint"} <= {r["Type"] for r in synth["Resources"].values()}

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()